# GAS-DSL vs Standard CUDA: Side-by-Side Kernel Verification
This notebook runs completely on **Google Colab (GPU)** with **zero OCaml / OPAM dependencies**.

For each graph algorithm:
1. **PTX Cell (`%%writefile <algo>.ptx`)**: Paste or inspect your generated PTX code directly. (Pre-filled with the current compiler output so you can run it immediately or swap in any new PTX).
2. **CUDA Cell (`%%writefile compare_<algo>.cu`)**: Standard handwritten CUDA C++ `__global__` kernel vs the PTX kernel loaded via the CUDA Driver API.
3. **Run Cell**: Compiles with `nvcc` and runs on the GPU, printing a **side-by-side comparison table of final values** for all vertices.


## 1. Verify GPU
Ensure your Colab runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU` or better).


In [ ]:
!nvidia-smi


---
## 1. Breadth-First Search (BFS)
Shortest hop distance using integer `Min` reduction.


### 1. PTX Content (`bfs.ptx`)
*Paste or modify your generated PTX code in this cell:*


In [ ]:
%%writefile bfs.ptx
.version 7.0
.target sm_75
.address_size 64

.visible .entry bfs_scatter(
    .param .u64 depth_ptr,
    .param .u64 row_offsets_ptr,
    .param .u64 src_indices_ptr,
    .param .u64 dst_indices_ptr,
    .param .u32 num_edges,
    .param .u32 num_vertices
)
{
    .reg .u32 %r<26>;
    .reg .u64 %rd<25>;
    .reg .pred %p<6>;
    mov.u32 %r0, %tid.x;
    mov.u32 %r1, %ntid.x;
    mov.u32 %r2, %ctaid.x;
    mad.lo.u32 %r3, %r2, %r1, %r0;
    ld.param.u32 %r4, [num_edges];
    setp.ge.u32 %p5, %r3, %r4;
    @%p5 bra $SCATTER_EXIT;
    ld.param.u64 %rd6, [src_indices_ptr];
    cvta.to.global.u64 %rd7, %rd6;
    ld.param.u64 %rd8, [dst_indices_ptr];
    cvta.to.global.u64 %rd9, %rd8;
    mul.wide.u32 %rd10, %r3, 4;
    add.u64 %rd11, %rd7, %rd10;
    ld.global.u32 %r12, [%rd11];
    mul.wide.u32 %rd13, %r3, 4;
    add.u64 %rd14, %rd9, %rd13;
    ld.global.u32 %r15, [%rd14];
    ld.param.u64 %rd16, [depth_ptr];
    cvta.to.global.u64 %rd17, %rd16;
    mul.wide.u32 %rd18, %r12, 4;
    add.u64 %rd19, %rd17, %rd18;
    ld.global.u32 %r20, [%rd19];
    add.u32 %r21, 1, 0;
    add.u32 %r22, %r20, %r21;
    mul.wide.u32 %rd23, %r15, 4;
    add.u64 %rd24, %rd17, %rd23;
    atom.global.min.u32 %r25, [%rd24], %r22;
$SCATTER_EXIT:
    ret;
}


### 2. Standard CUDA vs PTX Comparison Harness (`compare_bfs.cu`)
Contains the standard CUDA C++ `__global__` kernel and loads `bfs.ptx` via the CUDA Driver API.


In [ ]:
%%writefile compare_bfs.cu
#include <cuda.h>
#include <cuda_runtime.h>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <vector>
#include <string>
#include <fstream>
#include <sstream>
#include <iostream>
#include <iomanip>

#define CHECK_DRV(call) do { \
    CUresult err = (call); \
    if (err != CUDA_SUCCESS) { \
        const char *msg = nullptr; \
        cuGetErrorString(err, &msg); \
        fprintf(stderr, "CUDA Driver error %s:%d: %s\n", __FILE__, __LINE__, msg ? msg : "unknown"); \
        exit(1); \
    } \
} while(0)

#define CHECK_RT(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA Runtime error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(1); \
    } \
} while(0)

static std::string readFile(const std::string& path) {
    std::ifstream f(path.c_str());
    if (!f) { fprintf(stderr, "Failed to read file: %s\n", path.c_str()); exit(1); }
    std::stringstream ss; ss << f.rdbuf(); return ss.str();
}

struct Graph {
    uint32_t V, E;
    std::vector<uint32_t> row_offsets, src_indices, dst_indices;
    std::vector<uint32_t> w_u32;
    std::vector<float>    w_f32;
};

static Graph make_test_graph(uint32_t V = 256) {
    Graph g; g.V = V;
    g.row_offsets.assign(V + 1, 0);
    uint32_t seed = 42;
    auto next_rand = [&]() -> uint32_t { seed = seed * 1664525u + 1013904223u; return seed; };
    for (uint32_t u = 0; u < V; ++u) {
        g.row_offsets[u] = static_cast<uint32_t>(g.dst_indices.size());
        if (u + 1 < V) {
            g.src_indices.push_back(u); g.dst_indices.push_back(u + 1);
            g.w_u32.push_back((next_rand() % 10u) + 1u);
            g.w_f32.push_back(static_cast<float>((next_rand() % 10u) + 1u) * 0.5f);
        }
        for (int k = 0; k < 3; ++k) {
            g.src_indices.push_back(u); g.dst_indices.push_back(next_rand() % V);
            g.w_u32.push_back((next_rand() % 20u) + 1u);
            g.w_f32.push_back(static_cast<float>((next_rand() % 20u) + 1u) * 0.5f);
        }
    }
    g.row_offsets[V] = static_cast<uint32_t>(g.dst_indices.size());
    g.E = static_cast<uint32_t>(g.dst_indices.size());
    return g;
}

static void print_table_u32(const char* name, const uint32_t* ptx, const uint32_t* cuda, size_t n, size_t show = 10) {
    int mismatches = 0;
    for (size_t i = 0; i < n; ++i) if (ptx[i] != cuda[i]) mismatches++;
    std::cout << "\n======================================================================\n";
    std::cout << " Test: " << name << " (V=" << n << ")\n";
    std::cout << " Result: " << (mismatches == 0 ? "PASSED (EQUAL)" : "FAILED (MISMATCH)") << " | Mismatches: " << mismatches << "\n";
    std::cout << "----------------------------------------------------------------------\n";
    std::cout << std::setw(8) << "Vertex" << " | " << std::setw(18) << "GAS-DSL PTX Value" << " | " << std::setw(18) << "Standard CUDA Value" << " | Match?\n";
    std::cout << "----------------------------------------------------------------------\n";
    for (size_t i = 0; i < std::min(n, show); ++i) {
        std::cout << std::setw(8) << i << " | "
                  << std::setw(18) << ptx[i] << " | "
                  << std::setw(18) << cuda[i] << " | "
                  << (ptx[i] == cuda[i] ? " YES" : " NO") << "\n";
    }
    std::cout << "======================================================================\n\n";
}

static void print_table_f32(const char* name, const float* ptx, const float* cuda, size_t n, size_t show = 10, float tol = 1e-4f) {
    int mismatches = 0;
    for (size_t i = 0; i < n; ++i) if (std::fabs(ptx[i] - cuda[i]) > tol) mismatches++;
    std::cout << "\n======================================================================\n";
    std::cout << " Test: " << name << " (V=" << n << ")\n";
    std::cout << " Result: " << (mismatches == 0 ? "PASSED (EQUAL)" : "FAILED (MISMATCH)") << " | Mismatches: " << mismatches << "\n";
    std::cout << "----------------------------------------------------------------------\n";
    std::cout << std::setw(8) << "Vertex" << " | " << std::setw(18) << "GAS-DSL PTX Value" << " | " << std::setw(18) << "Standard CUDA Value" << " | Match?\n";
    std::cout << "----------------------------------------------------------------------\n";
    for (size_t i = 0; i < std::min(n, show); ++i) {
        std::cout << std::setw(8) << i << " | "
                  << std::setw(18) << std::fixed << std::setprecision(6) << ptx[i] << " | "
                  << std::setw(18) << std::fixed << std::setprecision(6) << cuda[i] << " | "
                  << (std::fabs(ptx[i] - cuda[i]) <= tol ? " YES" : " NO") << "\n";
    }
    std::cout << "======================================================================\n\n";
}

// Standard CUDA BFS Kernel
__global__ void cuda_bfs_kernel(uint32_t* depth, const uint32_t* row_offsets,
                                const uint32_t* src, const uint32_t* dst,
                                uint32_t num_edges, uint32_t num_vertices) {
    uint32_t e = blockIdx.x * blockDim.x + threadIdx.x;
    if (e < num_edges) {
        uint32_t u = src[e];
        uint32_t v = dst[e];
        atomicMin(&depth[v], depth[u] + 1);
    }
}

int main() {
    Graph g = make_test_graph();
    const uint32_t INF = 1000000000u;
    std::vector<uint32_t> init_depth(g.V, INF);
    init_depth[0] = 0;

    uint32_t *d_depth_cuda, *d_depth_ptx, *d_row_offsets, *d_src, *d_dst;
    CHECK_RT(cudaMalloc(&d_depth_cuda, g.V * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_depth_ptx,  g.V * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_row_offsets, (g.V + 1) * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_src, g.E * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_dst, g.E * sizeof(uint32_t)));

    CHECK_RT(cudaMemcpy(d_depth_cuda, init_depth.data(), g.V * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_depth_ptx,  init_depth.data(), g.V * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_row_offsets, g.row_offsets.data(), (g.V + 1) * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_src, g.src_indices.data(), g.E * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_dst, g.dst_indices.data(), g.E * sizeof(uint32_t), cudaMemcpyHostToDevice));

    // 1. Run Standard CUDA Kernel
    for (uint32_t iter = 0; iter < g.V - 1; ++iter) {
        cuda_bfs_kernel<<<(g.E + 255) / 256, 256>>>(d_depth_cuda, d_row_offsets, d_src, d_dst, g.E, g.V);
    }
    CHECK_RT(cudaDeviceSynchronize());
    std::vector<uint32_t> out_cuda(g.V);
    CHECK_RT(cudaMemcpy(out_cuda.data(), d_depth_cuda, g.V * sizeof(uint32_t), cudaMemcpyDeviceToHost));

    // 2. Run GAS-DSL PTX Kernel via Driver API
    CHECK_DRV(cuInit(0));
    CUdevice dev; CHECK_DRV(cuDeviceGet(&dev, 0));
    CUcontext ctx; CHECK_DRV(cuDevicePrimaryCtxRetain(&ctx, dev)); CHECK_DRV(cuCtxSetCurrent(ctx));
    CUmodule mod; CHECK_DRV(cuModuleLoadData(&mod, readFile("bfs.ptx").c_str()));
    CUfunction fn; CHECK_DRV(cuModuleGetFunction(&fn, mod, "bfs_scatter"));

    uint32_t num_edges = g.E, num_vertices = g.V;
    void* args[] = { &d_depth_ptx, &d_row_offsets, &d_src, &d_dst, &num_edges, &num_vertices };
    for (uint32_t iter = 0; iter < g.V - 1; ++iter) {
        CHECK_DRV(cuLaunchKernel(fn, (g.E + 255) / 256, 1, 1, 256, 1, 1, 0, nullptr, args, nullptr));
    }
    CHECK_DRV(cuCtxSynchronize());
    std::vector<uint32_t> out_ptx(g.V);
    CHECK_RT(cudaMemcpy(out_ptx.data(), d_depth_ptx, g.V * sizeof(uint32_t), cudaMemcpyDeviceToHost));

    // Print & Compare
    print_table_u32("BFS (Breadth-First Search)", out_ptx.data(), out_cuda.data(), g.V);

    cudaFree(d_depth_cuda); cudaFree(d_depth_ptx); cudaFree(d_row_offsets); cudaFree(d_src); cudaFree(d_dst);
    cuModuleUnload(mod);
    return 0;
}


### 3. Run on GPU & Print Final Values Table (`bfs`)


In [ ]:
!nvcc -O3 -arch=sm_75 compare_bfs.cu -o run_bfs -lcuda
!./run_bfs


---
## 2. In-Degree Count
Counts incoming edges per vertex using integer `Sum` atomic addition.


### 1. PTX Content (`in_degree.ptx`)
*Paste or modify your generated PTX code in this cell:*


In [ ]:
%%writefile in_degree.ptx
.version 7.0
.target sm_75
.address_size 64

.visible .entry in_degree_scatter(
    .param .u64 deg_ptr,
    .param .u64 row_offsets_ptr,
    .param .u64 src_indices_ptr,
    .param .u64 dst_indices_ptr,
    .param .u32 num_edges,
    .param .u32 num_vertices
)
{
    .reg .u32 %r<22>;
    .reg .u64 %rd<21>;
    .reg .pred %p<6>;
    mov.u32 %r0, %tid.x;
    mov.u32 %r1, %ntid.x;
    mov.u32 %r2, %ctaid.x;
    mad.lo.u32 %r3, %r2, %r1, %r0;
    ld.param.u32 %r4, [num_edges];
    setp.ge.u32 %p5, %r3, %r4;
    @%p5 bra $SCATTER_EXIT;
    ld.param.u64 %rd6, [src_indices_ptr];
    cvta.to.global.u64 %rd7, %rd6;
    ld.param.u64 %rd8, [dst_indices_ptr];
    cvta.to.global.u64 %rd9, %rd8;
    mul.wide.u32 %rd10, %r3, 4;
    add.u64 %rd11, %rd7, %rd10;
    ld.global.u32 %r12, [%rd11];
    mul.wide.u32 %rd13, %r3, 4;
    add.u64 %rd14, %rd9, %rd13;
    ld.global.u32 %r15, [%rd14];
    ld.param.u64 %rd16, [deg_ptr];
    cvta.to.global.u64 %rd17, %rd16;
    add.u32 %r18, 1, 0;
    mul.wide.u32 %rd19, %r15, 4;
    add.u64 %rd20, %rd17, %rd19;
    atom.global.add.u32 %r21, [%rd20], %r18;
$SCATTER_EXIT:
    ret;
}


### 2. Standard CUDA vs PTX Comparison Harness (`compare_in_degree.cu`)
Contains the standard CUDA C++ `__global__` kernel and loads `in_degree.ptx` via the CUDA Driver API.


In [ ]:
%%writefile compare_in_degree.cu
#include <cuda.h>
#include <cuda_runtime.h>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <vector>
#include <string>
#include <fstream>
#include <sstream>
#include <iostream>
#include <iomanip>

#define CHECK_DRV(call) do { \
    CUresult err = (call); \
    if (err != CUDA_SUCCESS) { \
        const char *msg = nullptr; \
        cuGetErrorString(err, &msg); \
        fprintf(stderr, "CUDA Driver error %s:%d: %s\n", __FILE__, __LINE__, msg ? msg : "unknown"); \
        exit(1); \
    } \
} while(0)

#define CHECK_RT(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA Runtime error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(1); \
    } \
} while(0)

static std::string readFile(const std::string& path) {
    std::ifstream f(path.c_str());
    if (!f) { fprintf(stderr, "Failed to read file: %s\n", path.c_str()); exit(1); }
    std::stringstream ss; ss << f.rdbuf(); return ss.str();
}

struct Graph {
    uint32_t V, E;
    std::vector<uint32_t> row_offsets, src_indices, dst_indices;
    std::vector<uint32_t> w_u32;
    std::vector<float>    w_f32;
};

static Graph make_test_graph(uint32_t V = 256) {
    Graph g; g.V = V;
    g.row_offsets.assign(V + 1, 0);
    uint32_t seed = 42;
    auto next_rand = [&]() -> uint32_t { seed = seed * 1664525u + 1013904223u; return seed; };
    for (uint32_t u = 0; u < V; ++u) {
        g.row_offsets[u] = static_cast<uint32_t>(g.dst_indices.size());
        if (u + 1 < V) {
            g.src_indices.push_back(u); g.dst_indices.push_back(u + 1);
            g.w_u32.push_back((next_rand() % 10u) + 1u);
            g.w_f32.push_back(static_cast<float>((next_rand() % 10u) + 1u) * 0.5f);
        }
        for (int k = 0; k < 3; ++k) {
            g.src_indices.push_back(u); g.dst_indices.push_back(next_rand() % V);
            g.w_u32.push_back((next_rand() % 20u) + 1u);
            g.w_f32.push_back(static_cast<float>((next_rand() % 20u) + 1u) * 0.5f);
        }
    }
    g.row_offsets[V] = static_cast<uint32_t>(g.dst_indices.size());
    g.E = static_cast<uint32_t>(g.dst_indices.size());
    return g;
}

static void print_table_u32(const char* name, const uint32_t* ptx, const uint32_t* cuda, size_t n, size_t show = 10) {
    int mismatches = 0;
    for (size_t i = 0; i < n; ++i) if (ptx[i] != cuda[i]) mismatches++;
    std::cout << "\n======================================================================\n";
    std::cout << " Test: " << name << " (V=" << n << ")\n";
    std::cout << " Result: " << (mismatches == 0 ? "PASSED (EQUAL)" : "FAILED (MISMATCH)") << " | Mismatches: " << mismatches << "\n";
    std::cout << "----------------------------------------------------------------------\n";
    std::cout << std::setw(8) << "Vertex" << " | " << std::setw(18) << "GAS-DSL PTX Value" << " | " << std::setw(18) << "Standard CUDA Value" << " | Match?\n";
    std::cout << "----------------------------------------------------------------------\n";
    for (size_t i = 0; i < std::min(n, show); ++i) {
        std::cout << std::setw(8) << i << " | "
                  << std::setw(18) << ptx[i] << " | "
                  << std::setw(18) << cuda[i] << " | "
                  << (ptx[i] == cuda[i] ? " YES" : " NO") << "\n";
    }
    std::cout << "======================================================================\n\n";
}

static void print_table_f32(const char* name, const float* ptx, const float* cuda, size_t n, size_t show = 10, float tol = 1e-4f) {
    int mismatches = 0;
    for (size_t i = 0; i < n; ++i) if (std::fabs(ptx[i] - cuda[i]) > tol) mismatches++;
    std::cout << "\n======================================================================\n";
    std::cout << " Test: " << name << " (V=" << n << ")\n";
    std::cout << " Result: " << (mismatches == 0 ? "PASSED (EQUAL)" : "FAILED (MISMATCH)") << " | Mismatches: " << mismatches << "\n";
    std::cout << "----------------------------------------------------------------------\n";
    std::cout << std::setw(8) << "Vertex" << " | " << std::setw(18) << "GAS-DSL PTX Value" << " | " << std::setw(18) << "Standard CUDA Value" << " | Match?\n";
    std::cout << "----------------------------------------------------------------------\n";
    for (size_t i = 0; i < std::min(n, show); ++i) {
        std::cout << std::setw(8) << i << " | "
                  << std::setw(18) << std::fixed << std::setprecision(6) << ptx[i] << " | "
                  << std::setw(18) << std::fixed << std::setprecision(6) << cuda[i] << " | "
                  << (std::fabs(ptx[i] - cuda[i]) <= tol ? " YES" : " NO") << "\n";
    }
    std::cout << "======================================================================\n\n";
}

// Standard CUDA In-Degree Kernel
__global__ void cuda_in_degree_kernel(uint32_t* deg, const uint32_t* row_offsets,
                                      const uint32_t* src, const uint32_t* dst,
                                      uint32_t num_edges, uint32_t num_vertices) {
    uint32_t e = blockIdx.x * blockDim.x + threadIdx.x;
    if (e < num_edges) {
        uint32_t v = dst[e];
        atomicAdd(&deg[v], 1u);
    }
}

int main() {
    Graph g = make_test_graph();
    std::vector<uint32_t> init_deg(g.V, 0);

    uint32_t *d_deg_cuda, *d_deg_ptx, *d_row_offsets, *d_src, *d_dst;
    CHECK_RT(cudaMalloc(&d_deg_cuda, g.V * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_deg_ptx,  g.V * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_row_offsets, (g.V + 1) * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_src, g.E * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_dst, g.E * sizeof(uint32_t)));

    CHECK_RT(cudaMemcpy(d_deg_cuda, init_deg.data(), g.V * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_deg_ptx,  init_deg.data(), g.V * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_row_offsets, g.row_offsets.data(), (g.V + 1) * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_src, g.src_indices.data(), g.E * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_dst, g.dst_indices.data(), g.E * sizeof(uint32_t), cudaMemcpyHostToDevice));

    // Standard CUDA
    cuda_in_degree_kernel<<<(g.E + 255) / 256, 256>>>(d_deg_cuda, d_row_offsets, d_src, d_dst, g.E, g.V);
    CHECK_RT(cudaDeviceSynchronize());
    std::vector<uint32_t> out_cuda(g.V);
    CHECK_RT(cudaMemcpy(out_cuda.data(), d_deg_cuda, g.V * sizeof(uint32_t), cudaMemcpyDeviceToHost));

    // GAS-DSL PTX
    CHECK_DRV(cuInit(0));
    CUdevice dev; CHECK_DRV(cuDeviceGet(&dev, 0));
    CUcontext ctx; CHECK_DRV(cuDevicePrimaryCtxRetain(&ctx, dev)); CHECK_DRV(cuCtxSetCurrent(ctx));
    CUmodule mod; CHECK_DRV(cuModuleLoadData(&mod, readFile("in_degree.ptx").c_str()));
    CUfunction fn; CHECK_DRV(cuModuleGetFunction(&fn, mod, "in_degree_scatter"));

    uint32_t num_edges = g.E, num_vertices = g.V;
    void* args[] = { &d_deg_ptx, &d_row_offsets, &d_src, &d_dst, &num_edges, &num_vertices };
    CHECK_DRV(cuLaunchKernel(fn, (g.E + 255) / 256, 1, 1, 256, 1, 1, 0, nullptr, args, nullptr));
    CHECK_DRV(cuCtxSynchronize());
    std::vector<uint32_t> out_ptx(g.V);
    CHECK_RT(cudaMemcpy(out_ptx.data(), d_deg_ptx, g.V * sizeof(uint32_t), cudaMemcpyDeviceToHost));

    print_table_u32("In-Degree Count", out_ptx.data(), out_cuda.data(), g.V);

    cudaFree(d_deg_cuda); cudaFree(d_deg_ptx); cudaFree(d_row_offsets); cudaFree(d_src); cudaFree(d_dst);
    cuModuleUnload(mod);
    return 0;
}


### 3. Run on GPU & Print Final Values Table (`in_degree`)


In [ ]:
!nvcc -O3 -arch=sm_75 compare_in_degree.cu -o run_in_degree -lcuda
!./run_in_degree


---
## 3. Single-Source Shortest Path (SSSP UInt32)
Bellman-Ford relaxation with integer weights using native PTX `atom.global.min.u32`.


### 1. PTX Content (`sssp_u32.ptx`)
*Paste or modify your generated PTX code in this cell:*


In [ ]:
%%writefile sssp_u32.ptx
.version 7.0
.target sm_75
.address_size 64

.visible .entry sssp_u32_scatter(
    .param .u64 dist_ptr,
    .param .u64 weight_ptr,
    .param .u64 row_offsets_ptr,
    .param .u64 src_indices_ptr,
    .param .u64 dst_indices_ptr,
    .param .u32 num_edges,
    .param .u32 num_vertices
)
{
    .reg .u32 %r<30>;
    .reg .u64 %rd<29>;
    .reg .pred %p<6>;
    mov.u32 %r0, %tid.x;
    mov.u32 %r1, %ntid.x;
    mov.u32 %r2, %ctaid.x;
    mad.lo.u32 %r3, %r2, %r1, %r0;
    ld.param.u32 %r4, [num_edges];
    setp.ge.u32 %p5, %r3, %r4;
    @%p5 bra $SCATTER_EXIT;
    ld.param.u64 %rd6, [src_indices_ptr];
    cvta.to.global.u64 %rd7, %rd6;
    ld.param.u64 %rd8, [dst_indices_ptr];
    cvta.to.global.u64 %rd9, %rd8;
    mul.wide.u32 %rd10, %r3, 4;
    add.u64 %rd11, %rd7, %rd10;
    ld.global.u32 %r12, [%rd11];
    mul.wide.u32 %rd13, %r3, 4;
    add.u64 %rd14, %rd9, %rd13;
    ld.global.u32 %r15, [%rd14];
    ld.param.u64 %rd16, [dist_ptr];
    cvta.to.global.u64 %rd17, %rd16;
    ld.param.u64 %rd18, [weight_ptr];
    cvta.to.global.u64 %rd19, %rd18;
    mul.wide.u32 %rd20, %r12, 4;
    add.u64 %rd21, %rd17, %rd20;
    ld.global.u32 %r22, [%rd21];
    mul.wide.u32 %rd23, %r3, 4;
    add.u64 %rd24, %rd19, %rd23;
    ld.global.u32 %r25, [%rd24];
    add.u32 %r26, %r22, %r25;
    mul.wide.u32 %rd27, %r15, 4;
    add.u64 %rd28, %rd17, %rd27;
    atom.global.min.u32 %r29, [%rd28], %r26;
$SCATTER_EXIT:
    ret;
}


### 2. Standard CUDA vs PTX Comparison Harness (`compare_sssp_u32.cu`)
Contains the standard CUDA C++ `__global__` kernel and loads `sssp_u32.ptx` via the CUDA Driver API.


In [ ]:
%%writefile compare_sssp_u32.cu
#include <cuda.h>
#include <cuda_runtime.h>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <vector>
#include <string>
#include <fstream>
#include <sstream>
#include <iostream>
#include <iomanip>

#define CHECK_DRV(call) do { \
    CUresult err = (call); \
    if (err != CUDA_SUCCESS) { \
        const char *msg = nullptr; \
        cuGetErrorString(err, &msg); \
        fprintf(stderr, "CUDA Driver error %s:%d: %s\n", __FILE__, __LINE__, msg ? msg : "unknown"); \
        exit(1); \
    } \
} while(0)

#define CHECK_RT(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA Runtime error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(1); \
    } \
} while(0)

static std::string readFile(const std::string& path) {
    std::ifstream f(path.c_str());
    if (!f) { fprintf(stderr, "Failed to read file: %s\n", path.c_str()); exit(1); }
    std::stringstream ss; ss << f.rdbuf(); return ss.str();
}

struct Graph {
    uint32_t V, E;
    std::vector<uint32_t> row_offsets, src_indices, dst_indices;
    std::vector<uint32_t> w_u32;
    std::vector<float>    w_f32;
};

static Graph make_test_graph(uint32_t V = 256) {
    Graph g; g.V = V;
    g.row_offsets.assign(V + 1, 0);
    uint32_t seed = 42;
    auto next_rand = [&]() -> uint32_t { seed = seed * 1664525u + 1013904223u; return seed; };
    for (uint32_t u = 0; u < V; ++u) {
        g.row_offsets[u] = static_cast<uint32_t>(g.dst_indices.size());
        if (u + 1 < V) {
            g.src_indices.push_back(u); g.dst_indices.push_back(u + 1);
            g.w_u32.push_back((next_rand() % 10u) + 1u);
            g.w_f32.push_back(static_cast<float>((next_rand() % 10u) + 1u) * 0.5f);
        }
        for (int k = 0; k < 3; ++k) {
            g.src_indices.push_back(u); g.dst_indices.push_back(next_rand() % V);
            g.w_u32.push_back((next_rand() % 20u) + 1u);
            g.w_f32.push_back(static_cast<float>((next_rand() % 20u) + 1u) * 0.5f);
        }
    }
    g.row_offsets[V] = static_cast<uint32_t>(g.dst_indices.size());
    g.E = static_cast<uint32_t>(g.dst_indices.size());
    return g;
}

static void print_table_u32(const char* name, const uint32_t* ptx, const uint32_t* cuda, size_t n, size_t show = 10) {
    int mismatches = 0;
    for (size_t i = 0; i < n; ++i) if (ptx[i] != cuda[i]) mismatches++;
    std::cout << "\n======================================================================\n";
    std::cout << " Test: " << name << " (V=" << n << ")\n";
    std::cout << " Result: " << (mismatches == 0 ? "PASSED (EQUAL)" : "FAILED (MISMATCH)") << " | Mismatches: " << mismatches << "\n";
    std::cout << "----------------------------------------------------------------------\n";
    std::cout << std::setw(8) << "Vertex" << " | " << std::setw(18) << "GAS-DSL PTX Value" << " | " << std::setw(18) << "Standard CUDA Value" << " | Match?\n";
    std::cout << "----------------------------------------------------------------------\n";
    for (size_t i = 0; i < std::min(n, show); ++i) {
        std::cout << std::setw(8) << i << " | "
                  << std::setw(18) << ptx[i] << " | "
                  << std::setw(18) << cuda[i] << " | "
                  << (ptx[i] == cuda[i] ? " YES" : " NO") << "\n";
    }
    std::cout << "======================================================================\n\n";
}

static void print_table_f32(const char* name, const float* ptx, const float* cuda, size_t n, size_t show = 10, float tol = 1e-4f) {
    int mismatches = 0;
    for (size_t i = 0; i < n; ++i) if (std::fabs(ptx[i] - cuda[i]) > tol) mismatches++;
    std::cout << "\n======================================================================\n";
    std::cout << " Test: " << name << " (V=" << n << ")\n";
    std::cout << " Result: " << (mismatches == 0 ? "PASSED (EQUAL)" : "FAILED (MISMATCH)") << " | Mismatches: " << mismatches << "\n";
    std::cout << "----------------------------------------------------------------------\n";
    std::cout << std::setw(8) << "Vertex" << " | " << std::setw(18) << "GAS-DSL PTX Value" << " | " << std::setw(18) << "Standard CUDA Value" << " | Match?\n";
    std::cout << "----------------------------------------------------------------------\n";
    for (size_t i = 0; i < std::min(n, show); ++i) {
        std::cout << std::setw(8) << i << " | "
                  << std::setw(18) << std::fixed << std::setprecision(6) << ptx[i] << " | "
                  << std::setw(18) << std::fixed << std::setprecision(6) << cuda[i] << " | "
                  << (std::fabs(ptx[i] - cuda[i]) <= tol ? " YES" : " NO") << "\n";
    }
    std::cout << "======================================================================\n\n";
}

// Standard CUDA SSSP u32 Kernel
__global__ void cuda_sssp_u32_kernel(uint32_t* dist, const uint32_t* weight,
                                     const uint32_t* row_offsets,
                                     const uint32_t* src, const uint32_t* dst,
                                     uint32_t num_edges, uint32_t num_vertices) {
    uint32_t e = blockIdx.x * blockDim.x + threadIdx.x;
    if (e < num_edges) {
        uint32_t u = src[e];
        uint32_t v = dst[e];
        atomicMin(&dist[v], dist[u] + weight[e]);
    }
}

int main() {
    Graph g = make_test_graph();
    const uint32_t INF = 1000000000u;
    std::vector<uint32_t> init_dist(g.V, INF);
    init_dist[0] = 0;

    uint32_t *d_dist_cuda, *d_dist_ptx, *d_weight, *d_row_offsets, *d_src, *d_dst;
    CHECK_RT(cudaMalloc(&d_dist_cuda, g.V * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_dist_ptx,  g.V * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_weight,    g.E * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_row_offsets, (g.V + 1) * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_src, g.E * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_dst, g.E * sizeof(uint32_t)));

    CHECK_RT(cudaMemcpy(d_dist_cuda, init_dist.data(), g.V * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_dist_ptx,  init_dist.data(), g.V * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_weight, g.w_u32.data(), g.E * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_row_offsets, g.row_offsets.data(), (g.V + 1) * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_src, g.src_indices.data(), g.E * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_dst, g.dst_indices.data(), g.E * sizeof(uint32_t), cudaMemcpyHostToDevice));

    // Standard CUDA
    for (uint32_t iter = 0; iter < g.V - 1; ++iter) {
        cuda_sssp_u32_kernel<<<(g.E + 255) / 256, 256>>>(d_dist_cuda, d_weight, d_row_offsets, d_src, d_dst, g.E, g.V);
    }
    CHECK_RT(cudaDeviceSynchronize());
    std::vector<uint32_t> out_cuda(g.V);
    CHECK_RT(cudaMemcpy(out_cuda.data(), d_dist_cuda, g.V * sizeof(uint32_t), cudaMemcpyDeviceToHost));

    // GAS-DSL PTX
    CHECK_DRV(cuInit(0));
    CUdevice dev; CHECK_DRV(cuDeviceGet(&dev, 0));
    CUcontext ctx; CHECK_DRV(cuDevicePrimaryCtxRetain(&ctx, dev)); CHECK_DRV(cuCtxSetCurrent(ctx));
    CUmodule mod; CHECK_DRV(cuModuleLoadData(&mod, readFile("sssp_u32.ptx").c_str()));
    CUfunction fn; CHECK_DRV(cuModuleGetFunction(&fn, mod, "sssp_u32_scatter"));

    uint32_t num_edges = g.E, num_vertices = g.V;
    void* args[] = { &d_dist_ptx, &d_weight, &d_row_offsets, &d_src, &d_dst, &num_edges, &num_vertices };
    for (uint32_t iter = 0; iter < g.V - 1; ++iter) {
        CHECK_DRV(cuLaunchKernel(fn, (g.E + 255) / 256, 1, 1, 256, 1, 1, 0, nullptr, args, nullptr));
    }
    CHECK_DRV(cuCtxSynchronize());
    std::vector<uint32_t> out_ptx(g.V);
    CHECK_RT(cudaMemcpy(out_ptx.data(), d_dist_ptx, g.V * sizeof(uint32_t), cudaMemcpyDeviceToHost));

    print_table_u32("SSSP (UInt32 Min)", out_ptx.data(), out_cuda.data(), g.V);

    cudaFree(d_dist_cuda); cudaFree(d_dist_ptx); cudaFree(d_weight);
    cudaFree(d_row_offsets); cudaFree(d_src); cudaFree(d_dst);
    cuModuleUnload(mod);
    return 0;
}


### 3. Run on GPU & Print Final Values Table (`sssp_u32`)


In [ ]:
!nvcc -O3 -arch=sm_75 compare_sssp_u32.cu -o run_sssp_u32 -lcuda
!./run_sssp_u32


---
## 4. Single-Source Shortest Path (SSSP Float32 with CAS)
Bellman-Ford relaxation with float weights using atomic CAS retry loop (`atom.global.cas.b32`).


### 1. PTX Content (`sssp_f32.ptx`)
*Paste or modify your generated PTX code in this cell:*


In [ ]:
%%writefile sssp_f32.ptx
.version 7.0
.target sm_75
.address_size 64

.visible .entry sssp_f32_scatter(
    .param .u64 dist_ptr,
    .param .u64 weight_ptr,
    .param .u64 row_offsets_ptr,
    .param .u64 src_indices_ptr,
    .param .u64 dst_indices_ptr,
    .param .u32 num_edges,
    .param .u32 num_vertices
)
{
    .reg .u32 %r<19>;
    .reg .u64 %rd<29>;
    .reg .f32 %f<30>;
    .reg .pred %p<32>;
    mov.u32 %r0, %tid.x;
    mov.u32 %r1, %ntid.x;
    mov.u32 %r2, %ctaid.x;
    mad.lo.u32 %r3, %r2, %r1, %r0;
    ld.param.u32 %r4, [num_edges];
    setp.ge.u32 %p5, %r3, %r4;
    @%p5 bra $SCATTER_EXIT;
    ld.param.u64 %rd6, [src_indices_ptr];
    cvta.to.global.u64 %rd7, %rd6;
    ld.param.u64 %rd8, [dst_indices_ptr];
    cvta.to.global.u64 %rd9, %rd8;
    mul.wide.u32 %rd10, %r3, 4;
    add.u64 %rd11, %rd7, %rd10;
    ld.global.u32 %r12, [%rd11];
    mul.wide.u32 %rd13, %r3, 4;
    add.u64 %rd14, %rd9, %rd13;
    ld.global.u32 %r15, [%rd14];
    ld.param.u64 %rd16, [dist_ptr];
    cvta.to.global.u64 %rd17, %rd16;
    ld.param.u64 %rd18, [weight_ptr];
    cvta.to.global.u64 %rd19, %rd18;
    mul.wide.u32 %rd20, %r12, 4;
    add.u64 %rd21, %rd17, %rd20;
    ld.global.f32 %f22, [%rd21];
    mul.wide.u32 %rd23, %r3, 4;
    add.u64 %rd24, %rd19, %rd23;
    ld.global.f32 %f25, [%rd24];
    add.f32 %f26, %f22, %f25;
    mul.wide.u32 %rd27, %r15, 4;
    add.u64 %rd28, %rd17, %rd27;
    mov.b32 %r16, %f26;
$CAS_LOOP_0:
    ld.global.f32 %f29, [%rd28];
    setp.le.f32 %p30, %f29, %f26;
    @%p30 bra $CAS_END_0;
    mov.b32 %r17, %f29;
    atom.global.cas.b32 %r18, [%rd28], %r17, %r16;
    setp.eq.b32 %p31, %r18, %r17;
    @!%p31 bra $CAS_LOOP_0;
$CAS_END_0:
$SCATTER_EXIT:
    ret;
}


### 2. Standard CUDA vs PTX Comparison Harness (`compare_sssp_f32.cu`)
Contains the standard CUDA C++ `__global__` kernel and loads `sssp_f32.ptx` via the CUDA Driver API.


In [ ]:
%%writefile compare_sssp_f32.cu
#include <cuda.h>
#include <cuda_runtime.h>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <vector>
#include <string>
#include <fstream>
#include <sstream>
#include <iostream>
#include <iomanip>

#define CHECK_DRV(call) do { \
    CUresult err = (call); \
    if (err != CUDA_SUCCESS) { \
        const char *msg = nullptr; \
        cuGetErrorString(err, &msg); \
        fprintf(stderr, "CUDA Driver error %s:%d: %s\n", __FILE__, __LINE__, msg ? msg : "unknown"); \
        exit(1); \
    } \
} while(0)

#define CHECK_RT(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA Runtime error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(1); \
    } \
} while(0)

static std::string readFile(const std::string& path) {
    std::ifstream f(path.c_str());
    if (!f) { fprintf(stderr, "Failed to read file: %s\n", path.c_str()); exit(1); }
    std::stringstream ss; ss << f.rdbuf(); return ss.str();
}

struct Graph {
    uint32_t V, E;
    std::vector<uint32_t> row_offsets, src_indices, dst_indices;
    std::vector<uint32_t> w_u32;
    std::vector<float>    w_f32;
};

static Graph make_test_graph(uint32_t V = 256) {
    Graph g; g.V = V;
    g.row_offsets.assign(V + 1, 0);
    uint32_t seed = 42;
    auto next_rand = [&]() -> uint32_t { seed = seed * 1664525u + 1013904223u; return seed; };
    for (uint32_t u = 0; u < V; ++u) {
        g.row_offsets[u] = static_cast<uint32_t>(g.dst_indices.size());
        if (u + 1 < V) {
            g.src_indices.push_back(u); g.dst_indices.push_back(u + 1);
            g.w_u32.push_back((next_rand() % 10u) + 1u);
            g.w_f32.push_back(static_cast<float>((next_rand() % 10u) + 1u) * 0.5f);
        }
        for (int k = 0; k < 3; ++k) {
            g.src_indices.push_back(u); g.dst_indices.push_back(next_rand() % V);
            g.w_u32.push_back((next_rand() % 20u) + 1u);
            g.w_f32.push_back(static_cast<float>((next_rand() % 20u) + 1u) * 0.5f);
        }
    }
    g.row_offsets[V] = static_cast<uint32_t>(g.dst_indices.size());
    g.E = static_cast<uint32_t>(g.dst_indices.size());
    return g;
}

static void print_table_u32(const char* name, const uint32_t* ptx, const uint32_t* cuda, size_t n, size_t show = 10) {
    int mismatches = 0;
    for (size_t i = 0; i < n; ++i) if (ptx[i] != cuda[i]) mismatches++;
    std::cout << "\n======================================================================\n";
    std::cout << " Test: " << name << " (V=" << n << ")\n";
    std::cout << " Result: " << (mismatches == 0 ? "PASSED (EQUAL)" : "FAILED (MISMATCH)") << " | Mismatches: " << mismatches << "\n";
    std::cout << "----------------------------------------------------------------------\n";
    std::cout << std::setw(8) << "Vertex" << " | " << std::setw(18) << "GAS-DSL PTX Value" << " | " << std::setw(18) << "Standard CUDA Value" << " | Match?\n";
    std::cout << "----------------------------------------------------------------------\n";
    for (size_t i = 0; i < std::min(n, show); ++i) {
        std::cout << std::setw(8) << i << " | "
                  << std::setw(18) << ptx[i] << " | "
                  << std::setw(18) << cuda[i] << " | "
                  << (ptx[i] == cuda[i] ? " YES" : " NO") << "\n";
    }
    std::cout << "======================================================================\n\n";
}

static void print_table_f32(const char* name, const float* ptx, const float* cuda, size_t n, size_t show = 10, float tol = 1e-4f) {
    int mismatches = 0;
    for (size_t i = 0; i < n; ++i) if (std::fabs(ptx[i] - cuda[i]) > tol) mismatches++;
    std::cout << "\n======================================================================\n";
    std::cout << " Test: " << name << " (V=" << n << ")\n";
    std::cout << " Result: " << (mismatches == 0 ? "PASSED (EQUAL)" : "FAILED (MISMATCH)") << " | Mismatches: " << mismatches << "\n";
    std::cout << "----------------------------------------------------------------------\n";
    std::cout << std::setw(8) << "Vertex" << " | " << std::setw(18) << "GAS-DSL PTX Value" << " | " << std::setw(18) << "Standard CUDA Value" << " | Match?\n";
    std::cout << "----------------------------------------------------------------------\n";
    for (size_t i = 0; i < std::min(n, show); ++i) {
        std::cout << std::setw(8) << i << " | "
                  << std::setw(18) << std::fixed << std::setprecision(6) << ptx[i] << " | "
                  << std::setw(18) << std::fixed << std::setprecision(6) << cuda[i] << " | "
                  << (std::fabs(ptx[i] - cuda[i]) <= tol ? " YES" : " NO") << "\n";
    }
    std::cout << "======================================================================\n\n";
}

// Helper: Standard Float CAS atomicMin
__device__ static void standard_atomicMinFloat(float* addr, float val) {
    int* addr_as_int = (int*)addr;
    int old = *addr_as_int, assumed;
    do {
        assumed = old;
        if (__int_as_float(assumed) <= val) break;
        old = atomicCAS(addr_as_int, assumed, __float_as_int(val));
    } while (assumed != old);
}

__global__ void cuda_sssp_f32_kernel(float* dist, const float* weight,
                                     const uint32_t* row_offsets,
                                     const uint32_t* src, const uint32_t* dst,
                                     uint32_t num_edges, uint32_t num_vertices) {
    uint32_t e = blockIdx.x * blockDim.x + threadIdx.x;
    if (e < num_edges) {
        uint32_t u = src[e];
        uint32_t v = dst[e];
        standard_atomicMinFloat(&dist[v], dist[u] + weight[e]);
    }
}

int main() {
    Graph g = make_test_graph();
    const float INF = 1e9f;
    std::vector<float> init_dist(g.V, INF);
    init_dist[0] = 0.0f;

    float *d_dist_cuda, *d_dist_ptx, *d_weight;
    uint32_t *d_row_offsets, *d_src, *d_dst;
    CHECK_RT(cudaMalloc(&d_dist_cuda, g.V * sizeof(float)));
    CHECK_RT(cudaMalloc(&d_dist_ptx,  g.V * sizeof(float)));
    CHECK_RT(cudaMalloc(&d_weight,    g.E * sizeof(float)));
    CHECK_RT(cudaMalloc(&d_row_offsets, (g.V + 1) * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_src, g.E * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_dst, g.E * sizeof(uint32_t)));

    CHECK_RT(cudaMemcpy(d_dist_cuda, init_dist.data(), g.V * sizeof(float), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_dist_ptx,  init_dist.data(), g.V * sizeof(float), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_weight, g.w_f32.data(), g.E * sizeof(float), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_row_offsets, g.row_offsets.data(), (g.V + 1) * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_src, g.src_indices.data(), g.E * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_dst, g.dst_indices.data(), g.E * sizeof(uint32_t), cudaMemcpyHostToDevice));

    // Standard CUDA
    for (uint32_t iter = 0; iter < g.V - 1; ++iter) {
        cuda_sssp_f32_kernel<<<(g.E + 255) / 256, 256>>>(d_dist_cuda, d_weight, d_row_offsets, d_src, d_dst, g.E, g.V);
    }
    CHECK_RT(cudaDeviceSynchronize());
    std::vector<float> out_cuda(g.V);
    CHECK_RT(cudaMemcpy(out_cuda.data(), d_dist_cuda, g.V * sizeof(float), cudaMemcpyDeviceToHost));

    // GAS-DSL PTX
    CHECK_DRV(cuInit(0));
    CUdevice dev; CHECK_DRV(cuDeviceGet(&dev, 0));
    CUcontext ctx; CHECK_DRV(cuDevicePrimaryCtxRetain(&ctx, dev)); CHECK_DRV(cuCtxSetCurrent(ctx));
    CUmodule mod; CHECK_DRV(cuModuleLoadData(&mod, readFile("sssp_f32.ptx").c_str()));
    CUfunction fn; CHECK_DRV(cuModuleGetFunction(&fn, mod, "sssp_f32_scatter"));

    uint32_t num_edges = g.E, num_vertices = g.V;
    void* args[] = { &d_dist_ptx, &d_weight, &d_row_offsets, &d_src, &d_dst, &num_edges, &num_vertices };
    for (uint32_t iter = 0; iter < g.V - 1; ++iter) {
        CHECK_DRV(cuLaunchKernel(fn, (g.E + 255) / 256, 1, 1, 256, 1, 1, 0, nullptr, args, nullptr));
    }
    CHECK_DRV(cuCtxSynchronize());
    std::vector<float> out_ptx(g.V);
    CHECK_RT(cudaMemcpy(out_ptx.data(), d_dist_ptx, g.V * sizeof(float), cudaMemcpyDeviceToHost));

    print_table_f32("SSSP (Float32 Min with CAS)", out_ptx.data(), out_cuda.data(), g.V);

    cudaFree(d_dist_cuda); cudaFree(d_dist_ptx); cudaFree(d_weight);
    cudaFree(d_row_offsets); cudaFree(d_src); cudaFree(d_dst);
    cuModuleUnload(mod);
    return 0;
}


### 3. Run on GPU & Print Final Values Table (`sssp_f32`)


In [ ]:
!nvcc -O3 -arch=sm_75 compare_sssp_f32.cu -o run_sssp_f32 -lcuda
!./run_sssp_f32


---
## 5. Max Influence (Float32 with CAS)
Propagates maximum influence scores using float atomic CAS max loop.


### 1. PTX Content (`max_influence.ptx`)
*Paste or modify your generated PTX code in this cell:*


In [ ]:
%%writefile max_influence.ptx
.version 7.0
.target sm_75
.address_size 64

.visible .entry max_influence_scatter(
    .param .u64 score_ptr,
    .param .u64 weight_ptr,
    .param .u64 row_offsets_ptr,
    .param .u64 src_indices_ptr,
    .param .u64 dst_indices_ptr,
    .param .u32 num_edges,
    .param .u32 num_vertices
)
{
    .reg .u32 %r<19>;
    .reg .u64 %rd<29>;
    .reg .f32 %f<30>;
    .reg .pred %p<32>;
    mov.u32 %r0, %tid.x;
    mov.u32 %r1, %ntid.x;
    mov.u32 %r2, %ctaid.x;
    mad.lo.u32 %r3, %r2, %r1, %r0;
    ld.param.u32 %r4, [num_edges];
    setp.ge.u32 %p5, %r3, %r4;
    @%p5 bra $SCATTER_EXIT;
    ld.param.u64 %rd6, [src_indices_ptr];
    cvta.to.global.u64 %rd7, %rd6;
    ld.param.u64 %rd8, [dst_indices_ptr];
    cvta.to.global.u64 %rd9, %rd8;
    mul.wide.u32 %rd10, %r3, 4;
    add.u64 %rd11, %rd7, %rd10;
    ld.global.u32 %r12, [%rd11];
    mul.wide.u32 %rd13, %r3, 4;
    add.u64 %rd14, %rd9, %rd13;
    ld.global.u32 %r15, [%rd14];
    ld.param.u64 %rd16, [score_ptr];
    cvta.to.global.u64 %rd17, %rd16;
    ld.param.u64 %rd18, [weight_ptr];
    cvta.to.global.u64 %rd19, %rd18;
    mul.wide.u32 %rd20, %r12, 4;
    add.u64 %rd21, %rd17, %rd20;
    ld.global.f32 %f22, [%rd21];
    mul.wide.u32 %rd23, %r3, 4;
    add.u64 %rd24, %rd19, %rd23;
    ld.global.f32 %f25, [%rd24];
    mul.f32 %f26, %f22, %f25;
    mul.wide.u32 %rd27, %r15, 4;
    add.u64 %rd28, %rd17, %rd27;
    mov.b32 %r16, %f26;
$CAS_LOOP_0:
    ld.global.f32 %f29, [%rd28];
    setp.ge.f32 %p30, %f29, %f26;
    @%p30 bra $CAS_END_0;
    mov.b32 %r17, %f29;
    atom.global.cas.b32 %r18, [%rd28], %r17, %r16;
    setp.eq.b32 %p31, %r18, %r17;
    @!%p31 bra $CAS_LOOP_0;
$CAS_END_0:
$SCATTER_EXIT:
    ret;
}


### 2. Standard CUDA vs PTX Comparison Harness (`compare_max_influence.cu`)
Contains the standard CUDA C++ `__global__` kernel and loads `max_influence.ptx` via the CUDA Driver API.


In [ ]:
%%writefile compare_max_influence.cu
#include <cuda.h>
#include <cuda_runtime.h>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <vector>
#include <string>
#include <fstream>
#include <sstream>
#include <iostream>
#include <iomanip>

#define CHECK_DRV(call) do { \
    CUresult err = (call); \
    if (err != CUDA_SUCCESS) { \
        const char *msg = nullptr; \
        cuGetErrorString(err, &msg); \
        fprintf(stderr, "CUDA Driver error %s:%d: %s\n", __FILE__, __LINE__, msg ? msg : "unknown"); \
        exit(1); \
    } \
} while(0)

#define CHECK_RT(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA Runtime error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(1); \
    } \
} while(0)

static std::string readFile(const std::string& path) {
    std::ifstream f(path.c_str());
    if (!f) { fprintf(stderr, "Failed to read file: %s\n", path.c_str()); exit(1); }
    std::stringstream ss; ss << f.rdbuf(); return ss.str();
}

struct Graph {
    uint32_t V, E;
    std::vector<uint32_t> row_offsets, src_indices, dst_indices;
    std::vector<uint32_t> w_u32;
    std::vector<float>    w_f32;
};

static Graph make_test_graph(uint32_t V = 256) {
    Graph g; g.V = V;
    g.row_offsets.assign(V + 1, 0);
    uint32_t seed = 42;
    auto next_rand = [&]() -> uint32_t { seed = seed * 1664525u + 1013904223u; return seed; };
    for (uint32_t u = 0; u < V; ++u) {
        g.row_offsets[u] = static_cast<uint32_t>(g.dst_indices.size());
        if (u + 1 < V) {
            g.src_indices.push_back(u); g.dst_indices.push_back(u + 1);
            g.w_u32.push_back((next_rand() % 10u) + 1u);
            g.w_f32.push_back(static_cast<float>((next_rand() % 10u) + 1u) * 0.5f);
        }
        for (int k = 0; k < 3; ++k) {
            g.src_indices.push_back(u); g.dst_indices.push_back(next_rand() % V);
            g.w_u32.push_back((next_rand() % 20u) + 1u);
            g.w_f32.push_back(static_cast<float>((next_rand() % 20u) + 1u) * 0.5f);
        }
    }
    g.row_offsets[V] = static_cast<uint32_t>(g.dst_indices.size());
    g.E = static_cast<uint32_t>(g.dst_indices.size());
    return g;
}

static void print_table_u32(const char* name, const uint32_t* ptx, const uint32_t* cuda, size_t n, size_t show = 10) {
    int mismatches = 0;
    for (size_t i = 0; i < n; ++i) if (ptx[i] != cuda[i]) mismatches++;
    std::cout << "\n======================================================================\n";
    std::cout << " Test: " << name << " (V=" << n << ")\n";
    std::cout << " Result: " << (mismatches == 0 ? "PASSED (EQUAL)" : "FAILED (MISMATCH)") << " | Mismatches: " << mismatches << "\n";
    std::cout << "----------------------------------------------------------------------\n";
    std::cout << std::setw(8) << "Vertex" << " | " << std::setw(18) << "GAS-DSL PTX Value" << " | " << std::setw(18) << "Standard CUDA Value" << " | Match?\n";
    std::cout << "----------------------------------------------------------------------\n";
    for (size_t i = 0; i < std::min(n, show); ++i) {
        std::cout << std::setw(8) << i << " | "
                  << std::setw(18) << ptx[i] << " | "
                  << std::setw(18) << cuda[i] << " | "
                  << (ptx[i] == cuda[i] ? " YES" : " NO") << "\n";
    }
    std::cout << "======================================================================\n\n";
}

static void print_table_f32(const char* name, const float* ptx, const float* cuda, size_t n, size_t show = 10, float tol = 1e-4f) {
    int mismatches = 0;
    for (size_t i = 0; i < n; ++i) if (std::fabs(ptx[i] - cuda[i]) > tol) mismatches++;
    std::cout << "\n======================================================================\n";
    std::cout << " Test: " << name << " (V=" << n << ")\n";
    std::cout << " Result: " << (mismatches == 0 ? "PASSED (EQUAL)" : "FAILED (MISMATCH)") << " | Mismatches: " << mismatches << "\n";
    std::cout << "----------------------------------------------------------------------\n";
    std::cout << std::setw(8) << "Vertex" << " | " << std::setw(18) << "GAS-DSL PTX Value" << " | " << std::setw(18) << "Standard CUDA Value" << " | Match?\n";
    std::cout << "----------------------------------------------------------------------\n";
    for (size_t i = 0; i < std::min(n, show); ++i) {
        std::cout << std::setw(8) << i << " | "
                  << std::setw(18) << std::fixed << std::setprecision(6) << ptx[i] << " | "
                  << std::setw(18) << std::fixed << std::setprecision(6) << cuda[i] << " | "
                  << (std::fabs(ptx[i] - cuda[i]) <= tol ? " YES" : " NO") << "\n";
    }
    std::cout << "======================================================================\n\n";
}

// Helper: Standard Float CAS atomicMax
__device__ static void standard_atomicMaxFloat(float* addr, float val) {
    int* addr_as_int = (int*)addr;
    int old = *addr_as_int, assumed;
    do {
        assumed = old;
        if (__int_as_float(assumed) >= val) break;
        old = atomicCAS(addr_as_int, assumed, __float_as_int(val));
    } while (assumed != old);
}

__global__ void cuda_max_influence_kernel(float* score, const float* weight,
                                          const uint32_t* row_offsets,
                                          const uint32_t* src, const uint32_t* dst,
                                          uint32_t num_edges, uint32_t num_vertices) {
    uint32_t e = blockIdx.x * blockDim.x + threadIdx.x;
    if (e < num_edges) {
        uint32_t u = src[e];
        uint32_t v = dst[e];
        standard_atomicMaxFloat(&score[v], score[u] * weight[e]);
    }
}

int main() {
    Graph g = make_test_graph();
    std::vector<float> init_score(g.V, 0.0f);
    init_score[0] = 100.0f;

    float *d_score_cuda, *d_score_ptx, *d_weight;
    uint32_t *d_row_offsets, *d_src, *d_dst;
    CHECK_RT(cudaMalloc(&d_score_cuda, g.V * sizeof(float)));
    CHECK_RT(cudaMalloc(&d_score_ptx,  g.V * sizeof(float)));
    CHECK_RT(cudaMalloc(&d_weight,     g.E * sizeof(float)));
    CHECK_RT(cudaMalloc(&d_row_offsets, (g.V + 1) * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_src, g.E * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_dst, g.E * sizeof(uint32_t)));

    CHECK_RT(cudaMemcpy(d_score_cuda, init_score.data(), g.V * sizeof(float), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_score_ptx,  init_score.data(), g.V * sizeof(float), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_weight, g.w_f32.data(), g.E * sizeof(float), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_row_offsets, g.row_offsets.data(), (g.V + 1) * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_src, g.src_indices.data(), g.E * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_dst, g.dst_indices.data(), g.E * sizeof(uint32_t), cudaMemcpyHostToDevice));

    // Standard CUDA
    for (unsigned iter = 0; iter < 10; ++iter) {
        cuda_max_influence_kernel<<<(g.E + 255) / 256, 256>>>(d_score_cuda, d_weight, d_row_offsets, d_src, d_dst, g.E, g.V);
    }
    CHECK_RT(cudaDeviceSynchronize());
    std::vector<float> out_cuda(g.V);
    CHECK_RT(cudaMemcpy(out_cuda.data(), d_score_cuda, g.V * sizeof(float), cudaMemcpyDeviceToHost));

    // GAS-DSL PTX
    CHECK_DRV(cuInit(0));
    CUdevice dev; CHECK_DRV(cuDeviceGet(&dev, 0));
    CUcontext ctx; CHECK_DRV(cuDevicePrimaryCtxRetain(&ctx, dev)); CHECK_DRV(cuCtxSetCurrent(ctx));
    CUmodule mod; CHECK_DRV(cuModuleLoadData(&mod, readFile("max_influence.ptx").c_str()));
    CUfunction fn; CHECK_DRV(cuModuleGetFunction(&fn, mod, "max_influence_scatter"));

    uint32_t num_edges = g.E, num_vertices = g.V;
    void* args[] = { &d_score_ptx, &d_weight, &d_row_offsets, &d_src, &d_dst, &num_edges, &num_vertices };
    for (unsigned iter = 0; iter < 10; ++iter) {
        CHECK_DRV(cuLaunchKernel(fn, (g.E + 255) / 256, 1, 1, 256, 1, 1, 0, nullptr, args, nullptr));
    }
    CHECK_DRV(cuCtxSynchronize());
    std::vector<float> out_ptx(g.V);
    CHECK_RT(cudaMemcpy(out_ptx.data(), d_score_ptx, g.V * sizeof(float), cudaMemcpyDeviceToHost));

    print_table_f32("Max Influence (Float32 Max with CAS)", out_ptx.data(), out_cuda.data(), g.V);

    cudaFree(d_score_cuda); cudaFree(d_score_ptx); cudaFree(d_weight);
    cudaFree(d_row_offsets); cudaFree(d_src); cudaFree(d_dst);
    cuModuleUnload(mod);
    return 0;
}


### 3. Run on GPU & Print Final Values Table (`max_influence`)


In [ ]:
!nvcc -O3 -arch=sm_75 compare_max_influence.cu -o run_max_influence -lcuda
!./run_max_influence


---
## 6. PageRank (Scatter + Apply)
Edge-parallel scatter (`atom.global.add.f32`) and vertex-parallel apply.


### 1. PTX Content (`pagerank.ptx`)
*Paste or modify your generated PTX code in this cell:*


In [ ]:
%%writefile pagerank.ptx
.version 7.0
.target sm_75
.address_size 64

.visible .entry pagerank_scatter(
    .param .u64 rank_ptr,
    .param .u64 out_deg_ptr,
    .param .u64 rank_accum_ptr,
    .param .u64 row_offsets_ptr,
    .param .u64 src_indices_ptr,
    .param .u64 dst_indices_ptr,
    .param .u32 num_edges,
    .param .u32 num_vertices
)
{
    .reg .u32 %r<16>;
    .reg .u64 %rd<31>;
    .reg .f32 %f<32>;
    .reg .pred %p<6>;
    mov.u32 %r0, %tid.x;
    mov.u32 %r1, %ntid.x;
    mov.u32 %r2, %ctaid.x;
    mad.lo.u32 %r3, %r2, %r1, %r0;
    ld.param.u32 %r4, [num_edges];
    setp.ge.u32 %p5, %r3, %r4;
    @%p5 bra $SCATTER_EXIT;
    ld.param.u64 %rd6, [src_indices_ptr];
    cvta.to.global.u64 %rd7, %rd6;
    ld.param.u64 %rd8, [dst_indices_ptr];
    cvta.to.global.u64 %rd9, %rd8;
    mul.wide.u32 %rd10, %r3, 4;
    add.u64 %rd11, %rd7, %rd10;
    ld.global.u32 %r12, [%rd11];
    mul.wide.u32 %rd13, %r3, 4;
    add.u64 %rd14, %rd9, %rd13;
    ld.global.u32 %r15, [%rd14];
    ld.param.u64 %rd16, [rank_ptr];
    cvta.to.global.u64 %rd17, %rd16;
    ld.param.u64 %rd18, [out_deg_ptr];
    cvta.to.global.u64 %rd19, %rd18;
    ld.param.u64 %rd20, [rank_accum_ptr];
    cvta.to.global.u64 %rd21, %rd20;
    mul.wide.u32 %rd22, %r12, 4;
    add.u64 %rd23, %rd17, %rd22;
    ld.global.f32 %f24, [%rd23];
    mul.wide.u32 %rd25, %r12, 4;
    add.u64 %rd26, %rd19, %rd25;
    ld.global.f32 %f27, [%rd26];
    div.rn.f32 %f28, %f24, %f27;
    mul.wide.u32 %rd29, %r15, 4;
    add.u64 %rd30, %rd21, %rd29;
    atom.global.add.f32 %f31, [%rd30], %f28;
$SCATTER_EXIT:
    ret;
}

.visible .entry pagerank_apply(
    .param .u64 rank_ptr,
    .param .u64 out_deg_ptr,
    .param .u64 rank_accum_ptr,
    .param .u32 num_vertices
)
{
    .reg .u32 %r<5>;
    .reg .u64 %rd<21>;
    .reg .f32 %f<19>;
    .reg .pred %p<6>;
    mov.u32 %r0, %tid.x;
    mov.u32 %r1, %ntid.x;
    mov.u32 %r2, %ctaid.x;
    mad.lo.u32 %r3, %r2, %r1, %r0;
    ld.param.u32 %r4, [num_vertices];
    setp.ge.u32 %p5, %r3, %r4;
    @%p5 bra $APPLY_EXIT;
    ld.param.u64 %rd6, [rank_ptr];
    cvta.to.global.u64 %rd7, %rd6;
    ld.param.u64 %rd8, [out_deg_ptr];
    cvta.to.global.u64 %rd9, %rd8;
    ld.param.u64 %rd10, [rank_accum_ptr];
    cvta.to.global.u64 %rd11, %rd10;
    add.f32 %f12, 0.150000, 0.0;
    add.f32 %f13, 0.850000, 0.0;
    mul.wide.u32 %rd14, %r3, 4;
    add.u64 %rd15, %rd11, %rd14;
    ld.global.f32 %f16, [%rd15];
    mul.f32 %f17, %f13, %f16;
    add.f32 %f18, %f12, %f17;
    mul.wide.u32 %rd19, %r3, 4;
    add.u64 %rd20, %rd7, %rd19;
    st.global.f32 [%rd20], %f18;
$APPLY_EXIT:
    ret;
}


### 2. Standard CUDA vs PTX Comparison Harness (`compare_pagerank.cu`)
Contains the standard CUDA C++ `__global__` kernel and loads `pagerank.ptx` via the CUDA Driver API.


In [ ]:
%%writefile compare_pagerank.cu
#include <cuda.h>
#include <cuda_runtime.h>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <vector>
#include <string>
#include <fstream>
#include <sstream>
#include <iostream>
#include <iomanip>

#define CHECK_DRV(call) do { \
    CUresult err = (call); \
    if (err != CUDA_SUCCESS) { \
        const char *msg = nullptr; \
        cuGetErrorString(err, &msg); \
        fprintf(stderr, "CUDA Driver error %s:%d: %s\n", __FILE__, __LINE__, msg ? msg : "unknown"); \
        exit(1); \
    } \
} while(0)

#define CHECK_RT(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA Runtime error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(1); \
    } \
} while(0)

static std::string readFile(const std::string& path) {
    std::ifstream f(path.c_str());
    if (!f) { fprintf(stderr, "Failed to read file: %s\n", path.c_str()); exit(1); }
    std::stringstream ss; ss << f.rdbuf(); return ss.str();
}

struct Graph {
    uint32_t V, E;
    std::vector<uint32_t> row_offsets, src_indices, dst_indices;
    std::vector<uint32_t> w_u32;
    std::vector<float>    w_f32;
};

static Graph make_test_graph(uint32_t V = 256) {
    Graph g; g.V = V;
    g.row_offsets.assign(V + 1, 0);
    uint32_t seed = 42;
    auto next_rand = [&]() -> uint32_t { seed = seed * 1664525u + 1013904223u; return seed; };
    for (uint32_t u = 0; u < V; ++u) {
        g.row_offsets[u] = static_cast<uint32_t>(g.dst_indices.size());
        if (u + 1 < V) {
            g.src_indices.push_back(u); g.dst_indices.push_back(u + 1);
            g.w_u32.push_back((next_rand() % 10u) + 1u);
            g.w_f32.push_back(static_cast<float>((next_rand() % 10u) + 1u) * 0.5f);
        }
        for (int k = 0; k < 3; ++k) {
            g.src_indices.push_back(u); g.dst_indices.push_back(next_rand() % V);
            g.w_u32.push_back((next_rand() % 20u) + 1u);
            g.w_f32.push_back(static_cast<float>((next_rand() % 20u) + 1u) * 0.5f);
        }
    }
    g.row_offsets[V] = static_cast<uint32_t>(g.dst_indices.size());
    g.E = static_cast<uint32_t>(g.dst_indices.size());
    return g;
}

static void print_table_u32(const char* name, const uint32_t* ptx, const uint32_t* cuda, size_t n, size_t show = 10) {
    int mismatches = 0;
    for (size_t i = 0; i < n; ++i) if (ptx[i] != cuda[i]) mismatches++;
    std::cout << "\n======================================================================\n";
    std::cout << " Test: " << name << " (V=" << n << ")\n";
    std::cout << " Result: " << (mismatches == 0 ? "PASSED (EQUAL)" : "FAILED (MISMATCH)") << " | Mismatches: " << mismatches << "\n";
    std::cout << "----------------------------------------------------------------------\n";
    std::cout << std::setw(8) << "Vertex" << " | " << std::setw(18) << "GAS-DSL PTX Value" << " | " << std::setw(18) << "Standard CUDA Value" << " | Match?\n";
    std::cout << "----------------------------------------------------------------------\n";
    for (size_t i = 0; i < std::min(n, show); ++i) {
        std::cout << std::setw(8) << i << " | "
                  << std::setw(18) << ptx[i] << " | "
                  << std::setw(18) << cuda[i] << " | "
                  << (ptx[i] == cuda[i] ? " YES" : " NO") << "\n";
    }
    std::cout << "======================================================================\n\n";
}

static void print_table_f32(const char* name, const float* ptx, const float* cuda, size_t n, size_t show = 10, float tol = 1e-4f) {
    int mismatches = 0;
    for (size_t i = 0; i < n; ++i) if (std::fabs(ptx[i] - cuda[i]) > tol) mismatches++;
    std::cout << "\n======================================================================\n";
    std::cout << " Test: " << name << " (V=" << n << ")\n";
    std::cout << " Result: " << (mismatches == 0 ? "PASSED (EQUAL)" : "FAILED (MISMATCH)") << " | Mismatches: " << mismatches << "\n";
    std::cout << "----------------------------------------------------------------------\n";
    std::cout << std::setw(8) << "Vertex" << " | " << std::setw(18) << "GAS-DSL PTX Value" << " | " << std::setw(18) << "Standard CUDA Value" << " | Match?\n";
    std::cout << "----------------------------------------------------------------------\n";
    for (size_t i = 0; i < std::min(n, show); ++i) {
        std::cout << std::setw(8) << i << " | "
                  << std::setw(18) << std::fixed << std::setprecision(6) << ptx[i] << " | "
                  << std::setw(18) << std::fixed << std::setprecision(6) << cuda[i] << " | "
                  << (std::fabs(ptx[i] - cuda[i]) <= tol ? " YES" : " NO") << "\n";
    }
    std::cout << "======================================================================\n\n";
}

// Standard CUDA PageRank Kernels
__global__ void cuda_pagerank_scatter(const float* rank, const float* out_deg, float* rank_accum,
                                      const uint32_t* row_offsets,
                                      const uint32_t* src, const uint32_t* dst,
                                      uint32_t num_edges, uint32_t num_vertices) {
    uint32_t e = blockIdx.x * blockDim.x + threadIdx.x;
    if (e < num_edges) {
        uint32_t u = src[e];
        uint32_t v = dst[e];
        atomicAdd(&rank_accum[v], rank[u] / out_deg[u]);
    }
}

__global__ void cuda_pagerank_apply(float* rank, const float* rank_accum, uint32_t num_vertices) {
    uint32_t v = blockIdx.x * blockDim.x + threadIdx.x;
    if (v < num_vertices) {
        rank[v] = 0.15f + 0.85f * rank_accum[v];
    }
}

int main() {
    Graph g = make_test_graph();
    std::vector<float> init_rank(g.V, 1.0f / g.V);
    std::vector<float> out_deg(g.V, 0.0f);
    for (uint32_t e = 0; e < g.E; ++e) out_deg[g.src_indices[e]] += 1.0f;
    for (uint32_t v = 0; v < g.V; ++v) if (out_deg[v] == 0.0f) out_deg[v] = 1.0f;

    float *d_rank_cuda, *d_rank_ptx, *d_out_deg, *d_accum_cuda, *d_accum_ptx;
    uint32_t *d_row_offsets, *d_src, *d_dst;
    CHECK_RT(cudaMalloc(&d_rank_cuda,  g.V * sizeof(float)));
    CHECK_RT(cudaMalloc(&d_rank_ptx,   g.V * sizeof(float)));
    CHECK_RT(cudaMalloc(&d_accum_cuda, g.V * sizeof(float)));
    CHECK_RT(cudaMalloc(&d_accum_ptx,  g.V * sizeof(float)));
    CHECK_RT(cudaMalloc(&d_out_deg,    g.V * sizeof(float)));
    CHECK_RT(cudaMalloc(&d_row_offsets, (g.V + 1) * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_src, g.E * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_dst, g.E * sizeof(uint32_t)));

    CHECK_RT(cudaMemcpy(d_rank_cuda, init_rank.data(), g.V * sizeof(float), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_rank_ptx,  init_rank.data(), g.V * sizeof(float), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_out_deg, out_deg.data(), g.V * sizeof(float), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_row_offsets, g.row_offsets.data(), (g.V + 1) * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_src, g.src_indices.data(), g.E * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_dst, g.dst_indices.data(), g.E * sizeof(uint32_t), cudaMemcpyHostToDevice));

    // Standard CUDA (10 iterations)
    for (int iter = 0; iter < 10; ++iter) {
        CHECK_RT(cudaMemset(d_accum_cuda, 0, g.V * sizeof(float)));
        cuda_pagerank_scatter<<<(g.E + 255) / 256, 256>>>(d_rank_cuda, d_out_deg, d_accum_cuda, d_row_offsets, d_src, d_dst, g.E, g.V);
        cuda_pagerank_apply<<<(g.V + 255) / 256, 256>>>(d_rank_cuda, d_accum_cuda, g.V);
    }
    CHECK_RT(cudaDeviceSynchronize());
    std::vector<float> out_cuda(g.V);
    CHECK_RT(cudaMemcpy(out_cuda.data(), d_rank_cuda, g.V * sizeof(float), cudaMemcpyDeviceToHost));

    // GAS-DSL PTX
    CHECK_DRV(cuInit(0));
    CUdevice dev; CHECK_DRV(cuDeviceGet(&dev, 0));
    CUcontext ctx; CHECK_DRV(cuDevicePrimaryCtxRetain(&ctx, dev)); CHECK_DRV(cuCtxSetCurrent(ctx));
    CUmodule mod; CHECK_DRV(cuModuleLoadData(&mod, readFile("pagerank.ptx").c_str()));
    CUfunction scatter_fn, apply_fn;
    CHECK_DRV(cuModuleGetFunction(&scatter_fn, mod, "pagerank_scatter"));
    CHECK_DRV(cuModuleGetFunction(&apply_fn, mod, "pagerank_apply"));

    uint32_t num_edges = g.E, num_vertices = g.V;
    void* s_args[] = { &d_rank_ptx, &d_out_deg, &d_accum_ptx, &d_row_offsets, &d_src, &d_dst, &num_edges, &num_vertices };
    void* a_args[] = { &d_rank_ptx, &d_out_deg, &d_accum_ptx, &num_vertices };

    for (int iter = 0; iter < 10; ++iter) {
        CHECK_RT(cudaMemset(d_accum_ptx, 0, g.V * sizeof(float)));
        CHECK_DRV(cuLaunchKernel(scatter_fn, (g.E + 255) / 256, 1, 1, 256, 1, 1, 0, nullptr, s_args, nullptr));
        CHECK_DRV(cuLaunchKernel(apply_fn,   (g.V + 255) / 256, 1, 1, 256, 1, 1, 0, nullptr, a_args, nullptr));
    }
    CHECK_DRV(cuCtxSynchronize());
    std::vector<float> out_ptx(g.V);
    CHECK_RT(cudaMemcpy(out_ptx.data(), d_rank_ptx, g.V * sizeof(float), cudaMemcpyDeviceToHost));

    print_table_f32("PageRank (Scatter + Apply)", out_ptx.data(), out_cuda.data(), g.V);

    cudaFree(d_rank_cuda); cudaFree(d_rank_ptx); cudaFree(d_accum_cuda); cudaFree(d_accum_ptx);
    cudaFree(d_out_deg); cudaFree(d_row_offsets); cudaFree(d_src); cudaFree(d_dst);
    cuModuleUnload(mod);
    return 0;
}


### 3. Run on GPU & Print Final Values Table (`pagerank`)


In [ ]:
!nvcc -O3 -arch=sm_75 compare_pagerank.cu -o run_pagerank -lcuda
!./run_pagerank


---
## 7. Weakly Connected Components (WCC)
Propagates minimum component IDs using integer `Min` atomic reduction.


### 1. PTX Content (`wcc.ptx`)
*Paste or modify your generated PTX code in this cell:*


In [ ]:
%%writefile wcc.ptx
.version 7.0
.target sm_75
.address_size 64

.visible .entry wcc_scatter(
    .param .u64 comp_ptr,
    .param .u64 row_offsets_ptr,
    .param .u64 src_indices_ptr,
    .param .u64 dst_indices_ptr,
    .param .u32 num_edges,
    .param .u32 num_vertices
)
{
    .reg .u32 %r<24>;
    .reg .u64 %rd<23>;
    .reg .pred %p<6>;
    mov.u32 %r0, %tid.x;
    mov.u32 %r1, %ntid.x;
    mov.u32 %r2, %ctaid.x;
    mad.lo.u32 %r3, %r2, %r1, %r0;
    ld.param.u32 %r4, [num_edges];
    setp.ge.u32 %p5, %r3, %r4;
    @%p5 bra $SCATTER_EXIT;
    ld.param.u64 %rd6, [src_indices_ptr];
    cvta.to.global.u64 %rd7, %rd6;
    ld.param.u64 %rd8, [dst_indices_ptr];
    cvta.to.global.u64 %rd9, %rd8;
    mul.wide.u32 %rd10, %r3, 4;
    add.u64 %rd11, %rd7, %rd10;
    ld.global.u32 %r12, [%rd11];
    mul.wide.u32 %rd13, %r3, 4;
    add.u64 %rd14, %rd9, %rd13;
    ld.global.u32 %r15, [%rd14];
    ld.param.u64 %rd16, [comp_ptr];
    cvta.to.global.u64 %rd17, %rd16;
    mul.wide.u32 %rd18, %r12, 4;
    add.u64 %rd19, %rd17, %rd18;
    ld.global.u32 %r20, [%rd19];
    mul.wide.u32 %rd21, %r15, 4;
    add.u64 %rd22, %rd17, %rd21;
    atom.global.min.u32 %r23, [%rd22], %r20;
$SCATTER_EXIT:
    ret;
}


### 2. Standard CUDA vs PTX Comparison Harness (`compare_wcc.cu`)
Contains the standard CUDA C++ `__global__` kernel and loads `wcc.ptx` via the CUDA Driver API.


In [ ]:
%%writefile compare_wcc.cu
#include <cuda.h>
#include <cuda_runtime.h>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <vector>
#include <string>
#include <fstream>
#include <sstream>
#include <iostream>
#include <iomanip>

#define CHECK_DRV(call) do { \
    CUresult err = (call); \
    if (err != CUDA_SUCCESS) { \
        const char *msg = nullptr; \
        cuGetErrorString(err, &msg); \
        fprintf(stderr, "CUDA Driver error %s:%d: %s\n", __FILE__, __LINE__, msg ? msg : "unknown"); \
        exit(1); \
    } \
} while(0)

#define CHECK_RT(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA Runtime error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(1); \
    } \
} while(0)

static std::string readFile(const std::string& path) {
    std::ifstream f(path.c_str());
    if (!f) { fprintf(stderr, "Failed to read file: %s\n", path.c_str()); exit(1); }
    std::stringstream ss; ss << f.rdbuf(); return ss.str();
}

struct Graph {
    uint32_t V, E;
    std::vector<uint32_t> row_offsets, src_indices, dst_indices;
    std::vector<uint32_t> w_u32;
    std::vector<float>    w_f32;
};

static Graph make_test_graph(uint32_t V = 256) {
    Graph g; g.V = V;
    g.row_offsets.assign(V + 1, 0);
    uint32_t seed = 42;
    auto next_rand = [&]() -> uint32_t { seed = seed * 1664525u + 1013904223u; return seed; };
    for (uint32_t u = 0; u < V; ++u) {
        g.row_offsets[u] = static_cast<uint32_t>(g.dst_indices.size());
        if (u + 1 < V) {
            g.src_indices.push_back(u); g.dst_indices.push_back(u + 1);
            g.w_u32.push_back((next_rand() % 10u) + 1u);
            g.w_f32.push_back(static_cast<float>((next_rand() % 10u) + 1u) * 0.5f);
        }
        for (int k = 0; k < 3; ++k) {
            g.src_indices.push_back(u); g.dst_indices.push_back(next_rand() % V);
            g.w_u32.push_back((next_rand() % 20u) + 1u);
            g.w_f32.push_back(static_cast<float>((next_rand() % 20u) + 1u) * 0.5f);
        }
    }
    g.row_offsets[V] = static_cast<uint32_t>(g.dst_indices.size());
    g.E = static_cast<uint32_t>(g.dst_indices.size());
    return g;
}

static void print_table_u32(const char* name, const uint32_t* ptx, const uint32_t* cuda, size_t n, size_t show = 10) {
    int mismatches = 0;
    for (size_t i = 0; i < n; ++i) if (ptx[i] != cuda[i]) mismatches++;
    std::cout << "\n======================================================================\n";
    std::cout << " Test: " << name << " (V=" << n << ")\n";
    std::cout << " Result: " << (mismatches == 0 ? "PASSED (EQUAL)" : "FAILED (MISMATCH)") << " | Mismatches: " << mismatches << "\n";
    std::cout << "----------------------------------------------------------------------\n";
    std::cout << std::setw(8) << "Vertex" << " | " << std::setw(18) << "GAS-DSL PTX Value" << " | " << std::setw(18) << "Standard CUDA Value" << " | Match?\n";
    std::cout << "----------------------------------------------------------------------\n";
    for (size_t i = 0; i < std::min(n, show); ++i) {
        std::cout << std::setw(8) << i << " | "
                  << std::setw(18) << ptx[i] << " | "
                  << std::setw(18) << cuda[i] << " | "
                  << (ptx[i] == cuda[i] ? " YES" : " NO") << "\n";
    }
    std::cout << "======================================================================\n\n";
}

static void print_table_f32(const char* name, const float* ptx, const float* cuda, size_t n, size_t show = 10, float tol = 1e-4f) {
    int mismatches = 0;
    for (size_t i = 0; i < n; ++i) if (std::fabs(ptx[i] - cuda[i]) > tol) mismatches++;
    std::cout << "\n======================================================================\n";
    std::cout << " Test: " << name << " (V=" << n << ")\n";
    std::cout << " Result: " << (mismatches == 0 ? "PASSED (EQUAL)" : "FAILED (MISMATCH)") << " | Mismatches: " << mismatches << "\n";
    std::cout << "----------------------------------------------------------------------\n";
    std::cout << std::setw(8) << "Vertex" << " | " << std::setw(18) << "GAS-DSL PTX Value" << " | " << std::setw(18) << "Standard CUDA Value" << " | Match?\n";
    std::cout << "----------------------------------------------------------------------\n";
    for (size_t i = 0; i < std::min(n, show); ++i) {
        std::cout << std::setw(8) << i << " | "
                  << std::setw(18) << std::fixed << std::setprecision(6) << ptx[i] << " | "
                  << std::setw(18) << std::fixed << std::setprecision(6) << cuda[i] << " | "
                  << (std::fabs(ptx[i] - cuda[i]) <= tol ? " YES" : " NO") << "\n";
    }
    std::cout << "======================================================================\n\n";
}

// Standard CUDA WCC Kernel
__global__ void cuda_wcc_kernel(uint32_t* comp, const uint32_t* row_offsets,
                                const uint32_t* src, const uint32_t* dst,
                                uint32_t num_edges, uint32_t num_vertices) {
    uint32_t e = blockIdx.x * blockDim.x + threadIdx.x;
    if (e < num_edges) {
        uint32_t u = src[e];
        uint32_t v = dst[e];
        atomicMin(&comp[v], comp[u]);
    }
}

int main() {
    Graph g = make_test_graph();
    std::vector<uint32_t> init_comp(g.V);
    for (uint32_t i = 0; i < g.V; ++i) init_comp[i] = i;

    uint32_t *d_comp_cuda, *d_comp_ptx, *d_row_offsets, *d_src, *d_dst;
    CHECK_RT(cudaMalloc(&d_comp_cuda, g.V * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_comp_ptx,  g.V * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_row_offsets, (g.V + 1) * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_src, g.E * sizeof(uint32_t)));
    CHECK_RT(cudaMalloc(&d_dst, g.E * sizeof(uint32_t)));

    CHECK_RT(cudaMemcpy(d_comp_cuda, init_comp.data(), g.V * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_comp_ptx,  init_comp.data(), g.V * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_row_offsets, g.row_offsets.data(), (g.V + 1) * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_src, g.src_indices.data(), g.E * sizeof(uint32_t), cudaMemcpyHostToDevice));
    CHECK_RT(cudaMemcpy(d_dst, g.dst_indices.data(), g.E * sizeof(uint32_t), cudaMemcpyHostToDevice));

    // Standard CUDA
    for (uint32_t iter = 0; iter < g.V - 1; ++iter) {
        cuda_wcc_kernel<<<(g.E + 255) / 256, 256>>>(d_comp_cuda, d_row_offsets, d_src, d_dst, g.E, g.V);
    }
    CHECK_RT(cudaDeviceSynchronize());
    std::vector<uint32_t> out_cuda(g.V);
    CHECK_RT(cudaMemcpy(out_cuda.data(), d_comp_cuda, g.V * sizeof(uint32_t), cudaMemcpyDeviceToHost));

    // GAS-DSL PTX
    CHECK_DRV(cuInit(0));
    CUdevice dev; CHECK_DRV(cuDeviceGet(&dev, 0));
    CUcontext ctx; CHECK_DRV(cuDevicePrimaryCtxRetain(&ctx, dev)); CHECK_DRV(cuCtxSetCurrent(ctx));
    CUmodule mod; CHECK_DRV(cuModuleLoadData(&mod, readFile("wcc.ptx").c_str()));
    CUfunction fn; CHECK_DRV(cuModuleGetFunction(&fn, mod, "wcc_scatter"));

    uint32_t num_edges = g.E, num_vertices = g.V;
    void* args[] = { &d_comp_ptx, &d_row_offsets, &d_src, &d_dst, &num_edges, &num_vertices };
    for (uint32_t iter = 0; iter < g.V - 1; ++iter) {
        CHECK_DRV(cuLaunchKernel(fn, (g.E + 255) / 256, 1, 1, 256, 1, 1, 0, nullptr, args, nullptr));
    }
    CHECK_DRV(cuCtxSynchronize());
    std::vector<uint32_t> out_ptx(g.V);
    CHECK_RT(cudaMemcpy(out_ptx.data(), d_comp_ptx, g.V * sizeof(uint32_t), cudaMemcpyDeviceToHost));

    print_table_u32("WCC (Weakly Connected Components)", out_ptx.data(), out_cuda.data(), g.V);

    cudaFree(d_comp_cuda); cudaFree(d_comp_ptx); cudaFree(d_row_offsets); cudaFree(d_src); cudaFree(d_dst);
    cuModuleUnload(mod);
    return 0;
}


### 3. Run on GPU & Print Final Values Table (`wcc`)


In [ ]:
!nvcc -O3 -arch=sm_75 compare_wcc.cu -o run_wcc -lcuda
!./run_wcc
